In [ ]:
import os
import torch
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional
from datasets import Dataset, ClassLabel
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline as hf_pipeline
)
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import HfApi, create_repo

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATA_DIR=Path("/content/drive/MyDrive/Colab Notebooks/Data/")

In [4]:
dtype_dict = {
    'customfield_12310921': 'object',
    'issuetype.subtask': 'object' # or 'object' if it contains mixed values
}

In [2]:
issues=pd.read_csv("C:\\Users\\qures\\MSAIJAN\\Programming for Artificial Intelligence\\filtered_issues.csv")
issues.shape

C:\Users\qures\AppData\Local\Temp\ipykernel_60132\1319884428.py:1: DtypeWarning: Columns (0: customfield_12310921) have mixed types. Specify dtype option on import or set low_memory=False.
  issues=pd.read_csv("C:\\Users\\qures\\MSAIJAN\\Programming for Artificial Intelligence\\filtered_issues.csv")


(496086, 37)

In [11]:
df=comments[comments['comment.author']=='7fb6e296']
df.sort_values(by='comment.updated')

,key,comment.id,comment.author,comment.body,comment.created,comment.updated
3252,AIRAVATA-2252,15752347,7fb6e296,Talked with Shameera. He's going to update the...,2016-12-15 20:53:51,2016-12-15 20:53:51


In [12]:
comments.isnull().sum()

key                0
comment.id         0
comment.author     0
comment.body       1
comment.created    0
comment.updated    0
dtype: int64

In [4]:
issues['issuetype.name'].unique()

<ArrowStringArray>
[                    'Bug',             'Improvement',
                    'Task',             'New Feature',
                    'Test',                    'Wish',
                'Sub-task',                     'RTC',
                'Question',                    'Temp',
           'TCK Challenge',      'Dependency upgrade',
                'Umbrella',                    'Epic',
          'Technical task',                   'Story',
           'Documentation',           'Brainstorming',
    'Suitable Name Search', 'Blog - New Blog Request',
            'New Git Repo',                 'Request',
                 'Project',          'Technical Debt',
              'Dependency',                 'Comment',
                  'Access',             'Requirement',
      'Github Integration',        'New JIRA Project',
                 'IT Help',                'Proposal',
            'Planned Work']
Length: 33, dtype: str

In [6]:
comments=pd.read_csv("C:\\Users\\qures\\MSAIJAN\\Programming for Artificial Intelligence\\filtered_jira_comments.csv")
comments.shape

(25000, 6)

In [ ]:
# --- 1. CORE PIPELINE ENGINE ---

class JiraPipeline:
    def __init__(self, ticket_id: str):
        self.ticket_id = ticket_id
        self.local_base = Path("cache")
        self.local_global = self.local_base / "global"
        self.local_ticket = self.local_base / "tickets" / ticket_id

        for p in [self.local_global, self.local_ticket]:
            p.mkdir(parents=True, exist_ok=True)

    def run_step(self, step_name: str, logic_func, is_global: bool = False) -> Any:
        folder = self.local_global if is_global else self.local_ticket
        path = folder / f"{step_name}.joblib"

        if path.exists():
            print(f"[CACHE] Step '{step_name}' hit.")
            return joblib.load(path)

        print(f"[RUN] Step '{step_name}' executing...")
        result = logic_func()
        joblib.dump(result, path)
        return result
    def create_model_registry():
        api = HfApi()
        username = api.whoami()["name"]
        repo_id = f"{username}/jira-ml-models"

        # Create repo if not exists
        create_repo(repo_id, exist_ok=True, repo_type="model")
        
        return api



In [ ]:



# --- 2. ML COMPONENTS ---

class PriorityPredictor:
    def __init__(self, model_path: str = None):
        self.model_path = model_path
        self.device = 0 if torch.cuda.is_available() else -1
        
    @staticmethod
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=1)
        return {
            "accuracy": accuracy_score(labels, predictions),
            "f1": f1_score(labels, predictions, average="weighted")
        }

    def train(self, df: pd.DataFrame, repo_id,hfAPI) -> Dict[str, Any]:
        """Returns metadata about the trained model."""
        save_path = "artifacts/global/priority_model_v1"

        # Prepare Dataset
        sample = df.sample(frac=0.2, random_state=42)
        dataset = Dataset.from_pandas(pd.DataFrame({
            'text': sample['summary'] + " " + sample['description'],
            'label': sample['priority.id']
        }))

        # Label Mapping logic
        unique_ids = sorted(set(dataset['label']))
        label_map = {old: i for i, old in enumerate(unique_ids)}
        dataset = dataset.map(lambda x: {'label': label_map[x['label']]})
        dataset = dataset.cast_column("label", ClassLabel(num_classes=len(unique_ids)))

        split = dataset.train_test_split(test_size=0.2)
        tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

        def tokenize_fn(batch):
            return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

        tokenized_ds = split.map(tokenize_fn, batched=True)
        tokenized_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

        model = DistilBertForSequenceClassification.from_pretrained(
            "distilbert-base-uncased", num_labels=len(unique_ids)
        )

        args = TrainingArguments(
            output_dir="./results",
            num_train_epochs=1,
            per_device_train_batch_size=16,
            max_steps=500,
            report_to="none"
        )

        trainer = Trainer(
            model=model, args=args, train_dataset=tokenized_ds["train"],
            eval_dataset=tokenized_ds["test"], compute_metrics=self.compute_metrics
        )

        trainer.train()
        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)
        # UPLOAD TO HF HUB
        hfAPI.upload_folder(
            folder_path=save_path,
            path_in_repo="priority_model_v1/",
            repo_id=repo_id,
            commit_message="Upload priority model"
        )
        return {"path": save_path, "labels": label_map, "num_labels": len(unique_ids)}

    def predict(self, text: str) -> Dict[str, Any]:
        classifier = hf_pipeline("text-classification", model=self.model_path, device=self.device)
        res = classifier(text)[0]
        return {"label": res['label'], "score": res['score']}





In [7]:
class SimilarTicketFinder:
    def __init__(self, index_path: str = None):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.index_path = index_path

    def fit(self, df: pd.DataFrame) -> str:
        embeddings = self.model.encode(df['text_combined'].tolist(), show_progress_bar=True)
        save_path = "artifacts/global/similarity_index.joblib"
        joblib.dump(embeddings, save_path)
        return save_path

    def find(self, text: str, historical_df: pd.DataFrame, top_k=5):
        embeddings = joblib.load(self.index_path)
        query_vec = self.model.encode([text])
        sims = cosine_similarity(query_vec, embeddings).flatten()
        idx = sims.argsort()[-top_k:][::-1]

        results = historical_df.iloc[idx].copy()
        results['similarity_score'] = sims[idx]
        return results

In [8]:
class DeveloperMoodChecker:
    def __init__(self):
        self.model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
        self.device = 0 if torch.cuda.is_available() else -1
        # Lazy load pipeline
        self.analyzer = hf_pipeline("sentiment-analysis", model=self.model_name, device=self.device)

    def analyze(self, comments_df: pd.DataFrame, developer: str) -> Dict[str, Any]:
        dev_comments = comments_df[comments_df['comment.author'] == developer]['comment.body'].tolist()
        if not dev_comments:
            return {"status": "No data"}

        # Batch inference (much faster)
        results = self.analyzer(dev_comments[:10], truncation=True)
        # map scores: positive (1) - negative (-1)
        scores = [1 if r['label'] == 'positive' else (-1 if r['label'] == 'negative' else 0) for r in results]
        avg = np.mean(scores)

        return {"avg_score": avg, "label": "Positive" if avg > 0.1 else "Neutral/Stressed"}





In [ ]:
# --- 3. AUTOMATION MANAGER ---

class JiraAutomation:
    def __init__(self, ticket_id: str):
        self.pipeline = JiraPipeline(ticket_id)
        self.issues_df = None
        self.comments_df = None


    def initialize_system(self):
        """Builds or loads the global brain."""
        # Step 1: Data
        data = self.pipeline.run_step("global_data", self._clean_data, is_global=True)
        self.issues_df = data['issues']
        self.comments_df = data['comments']

        if priority_model_path=="":
            
            # Step 2: Priority Model
            self.p_meta = self.pipeline.run_step(
                "priority_meta",
                lambda: PriorityPredictor().train(self.issues_df,repo_id,self.pipeline),
                is_global=True
            )
        else: 
            self.p_meta=priority_model_path
        # Step 3: Similarity Index
        self.s_path = self.pipeline.run_step(
            "similarity_path",
            lambda: SimilarTicketFinder().fit(self.issues_df),
            is_global=True
        )

    def _clean_data(self) -> Dict[str, pd.DataFrame]:
        df_issues=pd.read_csv(DATA_DIR/"filtered_issues.csv", dtype=dtype_dict)
        df_comments=pd.read_csv(DATA_DIR/"filtered_jira_comments.csv")

        # 2. Issues Cleaning Pipeline
        # Drop rows where summary is null (cannot predict without text)
        df_issues = df_issues.dropna(subset=['summary'])

        # Fill missing descriptions with summary to prevent 'NaN' strings
        df_issues['description'] = df_issues['description'].fillna(df_issues['summary'])

        # Feature Engineering: Combine text for BERT/Sentence-Transformers
        df_issues['text_combined'] = df_issues['summary'] + " " + df_issues['description']
        df_issues=df_issues[~df_issues['reporter'].isnull()]
        df_issues=df_issues[~df_issues['issuetype.description'].isnull()]
        df_issues=df_issues[~df_issues['priority.id'].isnull()]
        df_issues=df_issues[~df_issues['status.name'].isnull()]
        df_issues=df_issues[~df_issues['issuetype.name'].isnull()]
        df_issues=df_issues[~df_issues['assignee'].isnull()]


        # 3. Frequency-Based Filter (The 'Senior' way to handle noisy labels)
        # We only want to train on the most common priority IDs (1-5)
        counts = df_issues["priority.id"].value_counts()
        mainstream_ids = counts.head(5).index.tolist()
        df_issues = df_issues[df_issues["priority.id"].isin(mainstream_ids)]

        # 4. Column Selection (Dropping unnecessary noise to save memory/disk space)
        required_cols = [
            'key', 'text_combined', 'issuetype.name', 'project.key',
            'project.name', 'reporter', 'issuetype.description',
            'summary', 'description', 'assignee', 'status.name', 'priority.id'
        ]

        # Only keep columns that actually exist in the CSV to prevent KeyErrors
        existing_cols = [c for c in required_cols if c in df_issues.columns]
        df_issues_final = df_issues[existing_cols].copy()
        return {"issues": df_issues_final, "comments": df_comments}

    def process_new_ticket(self, summary: str, desc: str):
        """The Execution Loop."""
        full_text = f"{summary} {desc}"

        # 1. Predict
        predictor = PriorityPredictor(model_path=self.p_meta['path'])
        priority = self.pipeline.run_step("prediction", lambda: predictor.predict(full_text))

        # 2. Similar
        finder = SimilarTicketFinder(index_path=self.s_path)
        similar = self.pipeline.run_step("similar", lambda: finder.find(full_text, self.issues_df))

        # 3. Dev & Mood
        top_dev = similar.iloc[0]['assignee']
        mood_checker = DeveloperMoodChecker()
        mood = self.pipeline.run_step(f"mood_{top_dev}", lambda: mood_checker.analyze(self.comments_df, top_dev))

        return {
            "ticket": summary,
            "predicted_priority": priority,
            "recommended_dev": top_dev,
            "dev_mood": mood
        }

In [10]:
# --- 4. EXECUTION ---
if __name__ == "__main__":
    bot = JiraAutomation("PROJ-123")
    bot.initialize_system()
    final_report = bot.process_new_ticket("Login API Timeout", "Getting 504 Gateway errors on prod")
    print(final_report)

[RUN] Step 'global_data' executing...
[RUN] Step 'priority_meta' executing...


Map:   0%|          | 0/96090 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/96090 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[RUN] Step 'mood_7fb6e296' executing...
{'ticket': 'Login API Timeout', 'predicted_priority': {'label': 'LABEL_2', 'score': 0.7954341769218445}, 'recommended_dev': '7fb6e296', 'dev_mood': {'avg_score': np.float64(0.0), 'label': 'Neutral/Stressed'}}
